## Modelo: [MM19-MMGCN](https://github.com/weiyinwei/MMGCN)

In [1]:
import os, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import sys
sys.path.append('..')
from src.evaluation import temporal_train_test_split, evaluate_model
from src.text_features import TextFeatureExtractor
from src.image_features import CLIPFeatureExtractor

MODEL_NAME = 'mmgcn'
RESULTS_DIR = f'../results/{MODEL_NAME}'

# Hiperparámetros
EMBED_DIM = 64 # dim_E
DIM_LATENT_V = 256
AGGR_MODE = 'mean' # 'mean' o 'add'
CONCATE = False # concatenar h y x_hat (False = suma)
HAS_ID = True # residual id_embedding en cada capa
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-5 # reg_weight
BATCH_SIZE = 1024
LIKE_THRESHOLD = 4.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## Definición del modelo

Implementación fiel al repo [MM19-MMGCN](https://github.com/weiyinwei/MMGCN)

**Componentes:**
- `BaseModel`: capa de message passing (`h = aggr_neighbors(x @ W)`), equivalente a `BaseModel(MessagePassing)` del original.
- `GCN`: GCN por modalidad con 3 capas, embeddings `preference` por usuario, residual `id_embedding`, LeakyReLU y normalización L2 inicial.
- `Net`: un GCN por modalidad (imagen + texto), fusión por promedio aritmético, loss BPR + L2 reg.
- `TrainingDataset`: igual al original cada sample devuelve `([u,u], [pos,neg])`.

**Adaptaciones al repo original:**
1. Sin `torch_geometric`: adyacencia sparse normalizada con PyTorch puro.
2. 2 modalidades (imagen + texto) en lugar de 3 (video + audio + texto).
3. `preference` e `id_embedding` como `nn.Parameter` (el original los declara con `requires_grad=True` fuera de `nn.Parameter`, lo que en PyTorch moderno los excluiría del optimizer; aquí se hace explícito para que sí se optimicen).
4. Threshold de `like` para positivos en Yelp (feedback explícito).

In [ ]:
class BaseModel(nn.Module):
    """
    Equivalente sin PyG a BaseModel(MessagePassing) del repo original.
    forward: h = adj @ (x @ W)  con adj normalizada según aggr_mode.
    Init: uniform(-1/sqrt(in), 1/sqrt(in)) igual que torch_geometric.nn.inits.uniform.
    """
    def __init__(self, in_channels, out_channels, aggr='mean'):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(in_channels, out_channels))
        bound = 1.0 / math.sqrt(in_channels)
        nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, x, adj):
        x = x @ self.weight
        return torch.sparse.mm(adj, x)


class GCN(nn.Module):
    """
    GCN por modalidad. Fiel a la clase GCN del repo:
      - preference learnable por usuario
      - MLP opcional para proyectar features al espacio latente
      - 3 capas: conv_embed + linear (skip) + g_layer (combinación)
      - F.normalize al inicio, LeakyReLU en cada capa
      - id_embedding compartido como residual (si has_id=True)
    """
    def __init__(self, num_user, num_item, dim_feat, dim_id,
                 aggr_mode, concate, has_id, dim_latent=None):
        super().__init__()
        self.dim_latent = dim_latent
        self.concate = concate
        self.has_id = has_id

        dim0 = dim_latent if dim_latent else dim_feat  # dimensión tras MLP (o sin ella)

        if dim_latent:
            self.preference = nn.Parameter(
                nn.init.xavier_normal_(torch.rand(num_user, dim_latent))
            )
            self.MLP = nn.Linear(dim_feat, dim_latent)
        else:
            self.preference = nn.Parameter(
                nn.init.xavier_normal_(torch.rand(num_user, dim_feat))
            )

        self.conv_embed_1 = BaseModel(dim0, dim0, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_1.weight)
        self.linear_layer1 = nn.Linear(dim0, dim_id)
        nn.init.xavier_normal_(self.linear_layer1.weight)
        in1 = dim0 + dim_id if concate else dim0
        self.g_layer1 = nn.Linear(in1, dim_id)
        nn.init.xavier_normal_(self.g_layer1.weight)

        self.conv_embed_2 = BaseModel(dim_id, dim_id, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_2.weight)
        self.linear_layer2 = nn.Linear(dim_id, dim_id)
        nn.init.xavier_normal_(self.linear_layer2.weight)
        in2 = dim_id + dim_id if concate else dim_id
        self.g_layer2 = nn.Linear(in2, dim_id)
        nn.init.xavier_normal_(self.g_layer2.weight)

        self.conv_embed_3 = BaseModel(dim_id, dim_id, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_3.weight)
        self.linear_layer3 = nn.Linear(dim_id, dim_id)
        nn.init.xavier_normal_(self.linear_layer3.weight)
        in3 = dim_id + dim_id if concate else dim_id
        self.g_layer3 = nn.Linear(in3, dim_id)
        nn.init.xavier_normal_(self.g_layer3.weight)

    def _layer(self, conv, linear, g, x, id_emb, adj):
        h = F.leaky_relu(conv(x, adj))
        x_hat = F.leaky_relu(linear(x))
        if self.has_id:
            x_hat = x_hat + id_emb
        if self.concate:
            return F.leaky_relu(g(torch.cat([h, x_hat], dim=1)))
        return F.leaky_relu(g(h) + x_hat)

    def forward(self, features, id_embedding, adj):
        temp = self.MLP(features) if self.dim_latent else features
        x = torch.cat([self.preference, temp], dim=0)  # [N_users + N_items, dim0]
        x = F.normalize(x)  # L2, igual que el original

        x = self._layer(self.conv_embed_1, self.linear_layer1, self.g_layer1, x, id_embedding, adj)
        x = self._layer(self.conv_embed_2, self.linear_layer2, self.g_layer2, x, id_embedding, adj)
        x = self._layer(self.conv_embed_3, self.linear_layer3, self.g_layer3, x, id_embedding, adj)
        return x

In [3]:
class Net(nn.Module):
    """
    MMGCN fiel al Net del repo: un GCN por modalidad, fusión por promedio,
    id_embedding compartido, BPR loss + L2 regularización.

    Adaptación: 2 modalidades (img + text) en vez de 3 (v + a + t).
    Adyacencia construida como sparse tensor en vez de edge_index de PyG.
    """
    def __init__(self, img_feat, text_feat, edge_index,
                 num_user, num_item, aggr_mode, concate, has_id,
                 reg_weight, dim_x, device):
        super().__init__()
        self.num_user = num_user
        self.num_item = num_item
        self.reg_weight = reg_weight
        self.device = device
        # Vector BPR: matmul([pos, neg], [[1],[-1]]) = pos - neg
        self.register_buffer('bpr_weight', torch.tensor([[1.0], [-1.0]]))

        # --- Adyacencia bidireccional normalizada ---
        N = num_user + num_item
        ei = np.array(edge_index)            # [E, 2] con nodos ya offset
        rows = np.concatenate([ei[:, 0], ei[:, 1]])
        cols = np.concatenate([ei[:, 1], ei[:, 0]])
        deg = np.bincount(rows, minlength=N).astype(float)
        if aggr_mode == 'mean':
            vals = np.where(deg[rows] > 0, 1.0 / deg[rows], 0.0)
        else:
            vals = np.ones(len(rows), dtype=np.float32)
        idx = torch.LongTensor(np.stack([rows, cols]))
        v = torch.FloatTensor(vals)
        adj = torch.sparse_coo_tensor(idx, v, (N, N)).coalesce()
        self.register_buffer('adj', adj)

        # --- Features de ítems ---
        self.register_buffer('img_feat',  img_feat.float())
        self.register_buffer('text_feat', text_feat.float())

        # --- Un GCN por modalidad ---
        self.img_gcn = GCN(num_user, num_item, img_feat.shape[1],
                           dim_x, aggr_mode, concate, has_id, dim_latent=DIM_LATENT_V)
        self.text_gcn = GCN(num_user, num_item, text_feat.shape[1],
                            dim_x, aggr_mode, concate, has_id, dim_latent=None)

        # id_embedding compartido (residual en cada capa de ambas GCNs)
        self.id_embedding = nn.Parameter(
            nn.init.xavier_normal_(torch.rand(num_user + num_item, dim_x))
        )

    def forward(self):
        img_rep = self.img_gcn(self.img_feat,  self.id_embedding, self.adj)
        text_rep = self.text_gcn(self.text_feat, self.id_embedding, self.adj)
        return (img_rep + text_rep) / 2   # promedio (original: /3 para 3 modalidades)

    def loss(self, user_tensor, item_tensor):
        user_tensor = user_tensor.view(-1)
        item_tensor = item_tensor.view(-1)
        out = self.forward()

        user_score = out[user_tensor]
        item_score = out[item_tensor]
        score = torch.sum(user_score * item_score, dim=1).view(-1, 2)  # [B, 2]
        bpr_loss = -torch.mean(
            torch.log(torch.sigmoid(torch.matmul(score, self.bpr_weight)))
        )

        reg_emb = (
            (self.id_embedding[user_tensor]**2 + self.id_embedding[item_tensor]**2).mean()
            + (self.img_gcn.preference**2).mean()
        )
        return bpr_loss + self.reg_weight * reg_emb, bpr_loss

In [ ]:
class TrainingDataset(Dataset):
    def __init__(self, num_user, num_item, user_item_dict, pos_edges):
        self.pos_edges      = pos_edges        # list of (user_node, pos_item_node)
        self.user_item_dict = user_item_dict   # {user_node: set of item_nodes}
        self.all_items      = list(range(num_user, num_user + num_item))

    def __len__(self):
        return len(self.pos_edges)

    def __getitem__(self, index):
        user, pos_item = self.pos_edges[index]
        seen = self.user_item_dict.get(user, set())
        neg_item = random.choice(self.all_items)
        while neg_item in seen:
            neg_item = random.choice(self.all_items)
        return torch.LongTensor([user, user]), torch.LongTensor([pos_item, neg_item])

In [ ]:
class MMGCN:
    def __init__(self, text_embeddings, image_embeddings, embed_dim=64, device='cpu'):
        common = sorted(set(text_embeddings) & set(image_embeddings))
        self._item_ids = common
        self._item_idx = {b: i for i, b in enumerate(common)}
        self.num_item = len(common)

        self._img_feat = torch.FloatTensor(np.stack([image_embeddings[b] for b in common]))
        self._text_feat = torch.FloatTensor(np.stack([text_embeddings[b] for b in common]))

        self._embed_dim = embed_dim
        self.device = device
        self._net = None
        self._user_idx = None
        self.num_user = None
        self._emb_cache = None

        print(f'MMGCN_v2: {self.num_item:,} items | '
              f'image={self._img_feat.shape[1]}d | text={self._text_feat.shape[1]}d')

    def _build_structures(self, train_reviews):
        filtered = train_reviews[
            train_reviews['business_id'].isin(self._item_idx)
        ].copy()

        all_users = filtered['user_id'].unique()
        self._user_idx = {u: i for i, u in enumerate(all_users)}
        self.num_user = len(all_users)

        u_nodes = filtered['user_id'].map(self._user_idx).values
        i_nodes = filtered['business_id'].map(self._item_idx).values + self.num_user

        # Todas las interacciones para el grafo
        all_edges = list(zip(u_nodes.tolist(), i_nodes.tolist()))

        user_item_dict = defaultdict(set)
        for u, i in all_edges:
            user_item_dict[u].add(i)

        # Solo ítems "liked" como positivos para BPR (adaptación feedback explícito)
        liked = filtered[filtered['stars'] >= LIKE_THRESHOLD]
        lu = liked['user_id'].map(self._user_idx).values
        li = liked['business_id'].map(self._item_idx).values + self.num_user
        liked_edges = list(zip(lu.tolist(), li.tolist())) or all_edges

        print(f'  Usuarios: {self.num_user:,} | '
              f'Aristas grafo: {len(all_edges):,} | '
              f'Positivos (≥{LIKE_THRESHOLD}★): {len(liked_edges):,}')
        return all_edges, dict(user_item_dict), liked_edges

    def fit(self, train_reviews,
            epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE):
        torch.manual_seed(42)
        random.seed(42)
        self._emb_cache = None

        all_edges, user_item_dict, liked_edges = self._build_structures(train_reviews)

        self._net = Net(
            self._img_feat, self._text_feat, all_edges,
            self.num_user, self.num_item,
            aggr_mode=AGGR_MODE, concate=CONCATE, has_id=HAS_ID,
            reg_weight=weight_decay, dim_x=self._embed_dim, device=self.device
        ).to(self.device)

        dataset = TrainingDataset(self.num_user, self.num_item, user_item_dict, liked_edges)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        optimizer = torch.optim.Adam(self._net.parameters(), lr=lr)

        print(f'Train: {len(dataset):,} positivos | {epochs} epochs | bs={batch_size}')

        for epoch in range(1, epochs + 1):
            self._net.train()
            sum_loss = sum_bpr = steps = 0
            for u_t, i_t in loader:
                u_t, i_t = u_t.to(self.device), i_t.to(self.device)
                optimizer.zero_grad()
                loss, bpr_l = self._net.loss(u_t, i_t)
                loss.backward()
                optimizer.step()
                sum_loss += loss.item()
                sum_bpr += bpr_l.item()
                steps += 1
            if epoch % 10 == 0:
                print(f'  Epoch {epoch:3d}/{epochs} | '
                      f'loss: {sum_loss/steps:.4f} | bpr: {sum_bpr/steps:.4f}')

        self._net.eval()
        return self

    def recommend(self, user_id, train_reviews, top_k=10):
        if self._net is None or user_id not in self._user_idx:
            return []
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        cands = [b for b in self._item_ids if b not in seen]
        if not cands:
            return []

        if self._emb_cache is None:
            with torch.no_grad():
                self._emb_cache = self._net.forward().detach()

        u_node = self._user_idx[user_id]
        c_nodes = [self.num_user + self._item_idx[b] for b in cands]
        u_f = self._emb_cache[u_node].unsqueeze(0)
        c_f = self._emb_cache[c_nodes]
        scores = (u_f * c_f).sum(-1).cpu().numpy()

        top = np.argsort(scores)[::-1][:top_k]
        return [cands[i] for i in top]

## Datos y embeddings

In [6]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)

text_embeddings  = TextFeatureExtractor.load('text_embeddings.npz')
image_embeddings = CLIPFeatureExtractor.load('clip_embeddings.npz')

print(f'Text: {len(text_embeddings):,} | Image (CLIP): {len(image_embeddings):,}')
print(f'Overlap: {len(set(text_embeddings) & set(image_embeddings)):,} items con ambas modalidades')

Train: 83256 reviews | Test: 17191 reviews
Text: 1,151 | Image (CLIP): 3,824
Overlap: 1,151 items con ambas modalidades


## Entrenar MMGCN

In [7]:
model = MMGCN(
    text_embeddings, image_embeddings,
    embed_dim=EMBED_DIM, device=DEVICE,
)
model.fit(
    train_reviews,
    epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
)

MMGCN_v2: 1,151 items | image=512d | text=384d
  Usuarios: 10,490 | Aristas grafo: 83,256 | Positivos (≥4.0★): 62,155
Train: 62,155 positivos | 50 epochs | bs=1024
  Epoch  10/50 | loss: 0.1249 | bpr: 0.1249
  Epoch  20/50 | loss: 0.1040 | bpr: 0.1040
  Epoch  30/50 | loss: 0.0942 | bpr: 0.0942
  Epoch  40/50 | loss: 0.0864 | bpr: 0.0864
  Epoch  50/50 | loss: 0.0764 | bpr: 0.0764


## Evaluación

In [8]:
metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print('MMGCN metrics:')
print(metrics.round(4))

MMGCN metrics:
    precision  recall    ndcg
K                            
5      0.0286  0.0967  0.0686
10     0.0233  0.1538  0.0883
20     0.0185  0.2399  0.1123


## Guardar resultados

In [9]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump({
        'model': 'MMGCN',
        'embed_dim': EMBED_DIM,
        'dim_latent_v': DIM_LATENT_V,
        'aggr_mode': AGGR_MODE,
        'concate': CONCATE,
        'has_id': HAS_ID,
        'epochs': EPOCHS,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'batch_size': BATCH_SIZE,
        'like_threshold': LIKE_THRESHOLD,
        'image_model': 'CLIP (512d)',
        'text_model': 'all-MiniLM-L6-v2',
        'loss': 'BPR + L2 reg',
        'fusion': 'avg(img_rep, text_rep)',
        'gcn_layers': 3,
        'n_items_multimodal': model.num_item,
        'n_users': model.num_user,
    }, f, indent=2)
print(f'Saved -> results/{MODEL_NAME}/')

Saved -> results/mmgcn/
